# 05 · Evaluation metrics

Accuracy, per-class precision / recall, macro-F1 and confusion matrices (finalized gold labels only); evidence precision / recall against annotated evidence; unsupported-conclusion rate; latency, tokens and estimated cost — grouped by approach, company, claim category and difficulty. Missing measurements stay missing (`NaN` with the count measured), never filled in.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
RUN_ID = None
EXPERIMENT_ID = "claimverif-main-v1"

In [ ]:
from engine.metrics import evaluate, classification_metrics
from engine.notebook import banner, pick_run, describe_run, load_gold_labels, trace_index, save_table, save_figure
from engine.schema import LABELS
run = pick_run(EXPERIMENT_ID, RUN_ID)
assert run is not None, "no run found -- execute notebook 03 first"
describe_run(run)
preds, traces = run["predictions"], trace_index(run)
gold = load_gold_labels()
labelled = sum(1 for p in preds if p["claim_id"] in gold)
if not labelled:
    banner("None of this run's claims has a finalized gold label, so accuracy / precision / recall / macro-F1 / "
           "evidence metrics are not computed. Gold-free measures (unsupported conclusions, cost, latency) are shown.")

## By approach

In [ ]:
COLS = ["n_predictions", "n_labelled", "accuracy", "macro_f1", "evidence_precision", "evidence_recall", "n_evidence_scored",
        "decisive", "unsupported_conclusion_rate", "invalid_citations", "avg_latency_ms", "avg_input_tokens",
        "avg_output_tokens", "avg_llm_calls", "avg_cost_usd", "total_cost_usd", "n_cost", "is_mock"]
by_approach = pd.DataFrame(evaluate(preds, gold, traces, ("approach",)))
print("saved", save_table(by_approach, "metrics_by_approach", run))
by_approach[["approach"] + COLS]

## By company, claim category and difficulty

In [ ]:
for keys in (("approach", "company"), ("approach", "claim_category"), ("approach", "difficulty")):
    t = pd.DataFrame(evaluate(preds, gold, traces, keys))
    save_table(t, "metrics_by_" + "_".join(keys), run)
    display(t[list(keys) + ["n_predictions", "n_labelled", "accuracy", "macro_f1", "unsupported_conclusion_rate", "avg_cost_usd"]])

## Per-class precision / recall and confusion matrices

In [ ]:
if labelled:
    approaches = sorted({p["approach"] for p in preds})
    fig, axes = plt.subplots(1, len(approaches), figsize=(3.6 * len(approaches), 3.2), squeeze=False)
    for ax, a in zip(axes[0], approaches):
        ps = [p for p in preds if p["approach"] == a and p["claim_id"] in gold]
        m = classification_metrics([gold[p["claim_id"]]["label"] for p in ps], [p["predicted_label"] for p in ps])
        cm = pd.DataFrame(m["confusion_matrix"]).T
        display(pd.DataFrame(m["per_class"]).T.assign(approach=a))
        ax.imshow(cm.values, cmap="Blues"); ax.set_title(a, fontsize=9)
        ax.set_xticks(range(cm.shape[1])); ax.set_xticklabels([c[:4] for c in cm.columns], fontsize=7)
        ax.set_yticks(range(cm.shape[0])); ax.set_yticklabels([c[:4] for c in cm.index], fontsize=7)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, int(cm.values[i, j]), ha="center", va="center", fontsize=8)
    plt.tight_layout(); print("saved", save_figure(fig, f"05_confusion_{run['manifest']['run_id']}", run))
else:
    print("skipped: no labelled claims in this run")

## Self-check of the metric code (SYNTHETIC data, not research results)

Recomputes accuracy and macro-F1 by hand on synthetic labels and asserts the library agrees, so a reader can trust the tables above.

In [ ]:
from engine import synthetic
from engine.datasets import finalized_gold
sc = synthetic.synthetic_claims()
sa, sj = synthetic.synthetic_annotations(sc)
sgold = finalized_gold(sc, sa, sj)
spreds = synthetic.synthetic_predictions(sgold, {"A": 0.6, "B": 0.75})
res = {r["approach"]: r for r in evaluate(spreds, sgold, None, ("approach",))}
for a in ("A", "B"):
    ps = [p for p in spreds if p["approach"] == a]
    hand_acc = sum(p["predicted_label"] == sgold[p["claim_id"]]["label"] for p in ps) / len(ps)
    f1s = []
    for lab in LABELS:
        tp = sum(p["predicted_label"] == lab and sgold[p["claim_id"]]["label"] == lab for p in ps)
        fp = sum(p["predicted_label"] == lab and sgold[p["claim_id"]]["label"] != lab for p in ps)
        fn = sum(p["predicted_label"] != lab and sgold[p["claim_id"]]["label"] == lab for p in ps)
        if tp + fn:
            pr, rc = (tp / (tp + fp) if tp + fp else 0), tp / (tp + fn)
            f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0)
    assert abs(res[a]["accuracy"] - hand_acc) < 1e-12 and abs(res[a]["macro_f1"] - sum(f1s) / len(f1s)) < 1e-12
    print(f"synthetic {a}: accuracy {hand_acc:.3f}, macro-F1 {sum(f1s)/len(f1s):.3f} -- library matches hand computation")